# Seeing what happens in the network

A `Cluster` wires a node class into the simulator. After a run, `messages()` lists every message and its fate, `diagram()` draws a space-time diagram, `explain(node)` shows one node's view, and `timeline()` lists every event. Each of these renders as a table or a picture when it is the last expression in a cell; `print(...)` gives the plain-text version.

In [ ]:
from dslabs import Cluster, drop, partition
from dslabs.nodes import NodeMultiLeader

## One write, no faults

A client writes `x = 1` at n2. The naive multi-leader node applies it locally and broadcasts it. Time only advances when we say so.

In [ ]:
cluster = Cluster(NodeMultiLeader, 3, seed=1)
cluster.put("n2", "x", 1)
cluster.run_until(500)
cluster.values("x")

In [ ]:
cluster.messages()

In [ ]:
cluster.diagram()

## The same write with message loss

Half of all messages are dropped. The table names the rule that dropped each one, and `explain` answers the question a student will ask: why does n3 not have the value?

In [ ]:
cluster = Cluster(NodeMultiLeader, 3, seed=1)
cluster.add_rule(drop(0.5))
cluster.put("n2", "x", 1)
cluster.run_until(500)
cluster.values("x")

In [ ]:
cluster.messages()

In [ ]:
cluster.diagram()

In [ ]:
cluster.explain("n3")

## Reordering

Two writes 20 ms apart from different nodes. Latency varies per message, so a later write can arrive before an earlier one, and nodes end up disagreeing. Watch the arrows cross.

In [ ]:
cluster = Cluster(NodeMultiLeader, 3, seed=6, latency_ms=(30, 120))
cluster.put("n1", "x", 1)
cluster.run_until(20)
cluster.put("n3", "x", 2)
cluster.run_until(500)
cluster.values("x")

In [ ]:
cluster.diagram()

## A partition that heals

n1 is cut off for the first two seconds. Rules can be added and removed at any time, including from a timer.

In [ ]:
cluster = Cluster(NodeMultiLeader, 3, seed=2)
cut = partition({"n1"})
cluster.add_rule(cut)
cluster.scheduler.call_later(2000, lambda: cluster.remove_rule(cut))

cluster.put("n1", "x", 1)      # lost: n1 is isolated
cluster.run_until(2500)
cluster.put("n1", "x", 2)      # gets through
cluster.run_until(3000)
cluster.values("x")

In [ ]:
cluster.diagram()

In [ ]:
cluster.timeline()

## Every event, as text

The same information is available without a notebook: `print(cluster.messages())`, `print(cluster.timeline())`, `cluster.diagram().save("run.svg")`. Pass `verbose=True` to `Cluster` to see events printed as they happen.

In [ ]:
print(cluster.messages())